# LLM Inference Optimization

Serving an LLM is not like serving a classifier. The two phases of a request have opposite
performance characteristics, memory is the binding constraint rather than compute, and cost
scales with tokens rather than requests. This is the chapter that comes up when an interview
turns to "and how would you actually run this."

> ⏱️ Engine names and hardware numbers below were current in **August 2026** and will drift.
> The mechanics — prefill/decode asymmetry, batching, cache reuse — are durable.

## Why This Matters

- Prefill vs decode: why one is compute-bound and the other bandwidth-bound
- Continuous batching, and why it's the baseline rather than an optimization
- Paged and prefix-shared KV cache — the reason serving engines exist
- Quantization down to FP8/INT4, and what each level actually costs you
- Speculative decoding: the mechanism and its honest limits
- Reading a latency SLA in LLM terms: TTFT, TPOT, and end-to-end

## 1. Two Phases, Opposite Problems

```
PREFILL                                DECODE
Process the whole prompt at once       Emit one token at a time
All positions in parallel              Strictly sequential
Large matmuls -> COMPUTE-bound         One token's worth of math -> BANDWIDTH-bound
Scales with prompt length              Scales with output length
Determines TTFT                        Determines TPOT
```

**Why decode is bandwidth-bound.** To produce a single token you must read every weight in the
model plus the entire KV cache out of HBM, then perform roughly `2 × params` FLOPs. Modern
accelerators have far more arithmetic throughput than memory bandwidth, so the read dominates
and the compute units sit idle.

**Which is why batching is the whole game.** Reading the weights once and using them for 64
sequences amortizes the expensive part across 64 tokens. Batch size is the single biggest lever
on throughput — and it costs latency, which is the fundamental tension in LLM serving.

### The two latency metrics

| Metric | Meaning | Driven by |
|---|---|---|
| **TTFT** (time to first token) | Prompt in → first token out | Prefill; prompt length; queueing |
| **TPOT** (time per output token) | Steady-state inter-token gap | Decode; batch size; memory bandwidth |
| **End-to-end** | TTFT + TPOT × output tokens | Both |

A chat UI cares about TTFT (it feels responsive once streaming starts). A batch summarization
job cares only about end-to-end throughput. **These pull in opposite directions**, and "what's
your latency requirement" is really "which of these two."

In [ ]:
import numpy as np

def roofline(params_B, batch, ctx_len, bytes_per_param=1.0,
             hbm_tb_s=3.35, tflops=1979, mfu=0.35):
    """
    Rough prefill/decode characterization.
    Defaults are order-of-magnitude for a current-generation datacenter GPU
    (~3.3 TB/s HBM, ~2 PFLOPS dense FP8). Substitute your own hardware.
    """
    p = params_B * 1e9
    weight_bytes = p * bytes_per_param

    # DECODE: one token per sequence. Bandwidth-bound: read all weights once per step.
    decode_read_s = weight_bytes / (hbm_tb_s * 1e12)
    decode_flops = 2 * p * batch
    decode_compute_s = decode_flops / (tflops * 1e12 * mfu)

    # PREFILL: ctx_len tokens per sequence, all in parallel. Compute-bound.
    prefill_flops = 2 * p * batch * ctx_len
    prefill_compute_s = prefill_flops / (tflops * 1e12 * mfu)

    return {
        "decode_ms": max(decode_read_s, decode_compute_s) * 1000,
        "decode_bound": "memory" if decode_read_s > decode_compute_s else "compute",
        "decode_ratio": decode_read_s / decode_compute_s,
        "prefill_ms": max(prefill_compute_s, weight_bytes / (hbm_tb_s * 1e12)) * 1000,
    }

print("70B model, FP8 weights, 2k-token prompt\n")
print(f"{'batch':>6} {'decode/step':>13} {'bound':>8} {'read:compute':>14} {'prefill(all)':>14} {'tok/s':>9}")
print("-" * 66)
for b in [1, 8, 32, 128, 512]:
    r = roofline(70, batch=b, ctx_len=2048)
    tps = b / (r["decode_ms"] / 1000)
    print(f"{b:>6} {r['decode_ms']:>11.2f}ms {r['decode_bound']:>8} "
          f"{r['decode_ratio']:>13.1f}x {r['prefill_ms']:>12.0f}ms {tps:>9.0f}")

print()
print("Read the 'bound' column. At batch 1 the GPU spends ~100x longer reading weights")
print("than computing with them — you are renting a supercomputer to do memory copies.")
print("Extra sequences are nearly free until the ratio crosses 1 and you become")
print("compute-bound; past that, throughput stops improving and only latency grows.")
print("An under-batched deployment is the most common way to waste GPU money.")
print("(prefill column is for the whole batch, which is why it scales with it)")

## 2. Continuous Batching

**Static batching** collects N requests, runs them together until *all* finish, then starts
the next batch. Since output lengths vary wildly, short requests sit idle waiting for the
longest one, and no new work is admitted meanwhile.

**Continuous batching** (also "in-flight batching") makes the admission decision at *every
forward step*: finished sequences are evicted immediately and queued requests take their slot.

This is not an optimization you tune — it has been table stakes in every serious engine since
about 2023. It's worth knowing precisely because an interviewer may probe whether you think
it's still a differentiator.

In [ ]:
rng = np.random.default_rng(7)

def simulate(requests, max_concurrent, mode, step_ms=12.0):
    """
    requests: list of output lengths (in tokens).
    Returns mean completion latency in ms.
    """
    n = len(requests)
    done = np.zeros(n)
    if mode == "static":
        t = 0.0
        for i in range(0, n, max_concurrent):
            grp = requests[i:i + max_concurrent]
            batch_ms = max(grp) * step_ms          # everyone waits for the longest
            for j, _ in enumerate(grp):
                done[i + j] = t + batch_ms
            t += batch_ms
    else:  # continuous
        remaining = list(requests)
        idx = list(range(n))
        active, t = [], 0.0
        while remaining or active:
            while len(active) < max_concurrent and remaining:
                active.append([idx.pop(0), remaining.pop(0)])
            t += step_ms
            for slot in active:
                slot[1] -= 1
            for slot in [s for s in active if s[1] <= 0]:
                done[slot[0]] = t                   # evicted the moment it finishes
                active.remove(slot)
    return done.mean(), done.max()

# Realistic workload: most replies short, a few very long.
lengths = np.concatenate([
    rng.integers(20, 80, size=180),
    rng.integers(400, 900, size=20),
]).tolist()
rng.shuffle(lengths)

print(f"{'scheduling':<16}{'mean latency':>15}{'makespan':>12}")
print("-" * 43)
for mode in ["static", "continuous"]:
    mean_ms, max_ms = simulate(lengths, max_concurrent=16, mode=mode)
    print(f"{mode:<16}{mean_ms/1000:>13.1f}s {max_ms/1000:>10.1f}s")

s_mean, _ = simulate(lengths, 16, "static")
c_mean, _ = simulate(lengths, 16, "continuous")
print(f"\nMean latency improvement: {s_mean / c_mean:.1f}x")
print("The gain comes entirely from output-length variance. If every request emitted")
print("exactly the same number of tokens, the two schedulers would be identical.")

## 3. KV Cache Management

The KV cache is the reason a general web server can't just "run the model." Two ideas define
the modern serving stack:

### PagedAttention
Naively, you reserve contiguous memory for each sequence's maximum possible length. Since
actual lengths vary enormously, most of that reservation is never used — internal fragmentation
that wastes most of the cache.

PagedAttention borrows the operating-system idea: store KV in **fixed-size pages** with an
indirection table mapping logical positions to physical pages. Allocate pages on demand, so
waste is bounded by one partial page per sequence. This is what let batch sizes grow by an
order of magnitude, and it's the core idea behind vLLM.

### Prefix caching / RadixAttention
If many requests share a prefix — the same system prompt, the same few-shot examples, the same
document in a RAG pipeline, the same conversation history on each agent turn — the KV for that
prefix is *identical* every time. Cache it and reuse it.

RadixAttention organizes cached prefixes in a radix tree so overlaps are found automatically
rather than declared. For agent workloads, where each turn re-sends a growing history, this is
frequently the single largest win available.

> 💡 **Interview Tip:** "We'd put the invariant content at the front of the prompt so it's
> prefix-cacheable" is a small sentence that signals real serving experience. It's also
> actionable advice: reordering a prompt to make the stable part a literal prefix can cut TTFT
> dramatically at no quality cost.

In [ ]:
def fragmentation(seq_lens, max_len, page_size=16):
    """Compare naive max-length reservation against paged allocation."""
    naive = len(seq_lens) * max_len
    paged = sum(int(np.ceil(L / page_size)) * page_size for L in seq_lens)
    used = sum(seq_lens)
    return used, naive, paged

lens = rng.integers(50, 600, size=64).tolist()
used, naive, paged = fragmentation(lens, max_len=4096)

print("64 concurrent sequences, actual lengths 50-600, max supported 4096\n")
print(f"{'scheme':<26}{'KV slots':>12}{'utilization':>14}")
print("-" * 52)
print(f"{'actual tokens':<26}{used:>12,}{'—':>14}")
print(f"{'naive max reservation':<26}{naive:>12,}{used/naive:>13.1%}")
print(f"{'paged (page=16)':<26}{paged:>12,}{used/paged:>13.1%}")
print(f"\nPaging fits {naive/paged:.1f}x more sequences in the same memory.")

# --- Prefix caching on an agent workload ---
print("\n\nPrefix reuse across agent turns (2k-token system prompt + growing history):\n")
SYSTEM = 2000
print(f"{'turn':>5}{'total prompt':>14}{'no cache':>11}{'w/ prefix cache':>17}")
print("-" * 48)
total_nocache = total_cache = 0
history = 0
for turn in range(1, 7):
    prompt = SYSTEM + history
    new_tokens = prompt if turn == 1 else prompt - prev_prompt
    total_nocache += prompt
    total_cache += new_tokens
    print(f"{turn:>5}{prompt:>14,}{prompt:>11,}{new_tokens:>17,}")
    prev_prompt = prompt
    history += 350                      # each turn appends output + next user message

print("-" * 48)
print(f"{'TOTAL':>5}{'':>14}{total_nocache:>11,}{total_cache:>17,}")
print(f"\nPrefill tokens saved: {1 - total_cache/total_nocache:.0%}")
print("This is why agent frameworks obsess over keeping the prompt prefix byte-identical.")
print("A single changing token near the front (a timestamp, a shuffled tool list)")
print("invalidates the entire cache below it.")

## 4. Quantization

| Precision | Bytes/param | 70B weights | Typical quality cost | Notes |
|---|---|---|---|---|
| **FP32** | 4 | 280 GB | — | Training only; nobody serves this |
| **BF16** | 2 | 140 GB | Reference | Long-time default; still the quality baseline |
| **FP8** | 1 | 70 GB | Very small | Native on recent datacenter GPUs; now common for weights *and* KV cache |
| **INT8** | 1 | 70 GB | Small | Mature tooling; broad hardware support |
| **INT4** | 0.5 | 35 GB | Noticeable, task-dependent | AWQ / GPTQ; strong for memory-bound single-stream |
| **Sub-4-bit** | <0.5 | <35 GB | Significant | Edge and local use |

**Two things to be precise about:**

1. **Weight quantization and KV-cache quantization are separate decisions.** You can serve
   BF16 weights with an FP8 cache, or vice versa. At long context the cache often dominates
   memory, so quantizing it can matter more than quantizing weights.
2. **Quantization mainly buys memory and bandwidth, not FLOPs** — unless the hardware has
   native support for that format's arithmetic. Since decode is bandwidth-bound, halving bytes
   read roughly halves decode time, which is why it feels like a compute win.

**Evaluate on your task, not on perplexity.** Quantization damage is uneven: it often shows up
first in long-context retrieval, code generation, and multi-step reasoning while leaving
short-form benchmarks nearly untouched.

## 5. Speculative Decoding

A small **draft** model proposes K tokens cheaply. The large **verifier** scores all K+1
positions in a *single* forward pass — which costs barely more than one token, because that
pass is parallel like prefill. Accept the longest prefix the verifier agrees with, then
correct at the first divergence.

**Why it's free quality-wise:** with the standard acceptance rule, the output distribution is
mathematically identical to sampling from the large model alone. You are trading compute for
latency, not quality for latency.

**Where it disappoints:**
- Speedup is entirely governed by the acceptance rate. Predictable text (boilerplate, code
  scaffolding, structured output) accepts well; creative or high-entropy text does not.
- It burns extra FLOPs. Under a heavily batched, compute-saturated server it can *reduce*
  aggregate throughput even while improving single-stream latency.
- You now operate two models, and the draft must stay aligned with the verifier across updates.

**Variants worth naming:** self-speculation using early layers as the draft; n-gram or
prompt-lookup decoding (draft by copying from the prompt — excellent for summarization and
editing, where output heavily overlaps input); and Medusa-style multi-head drafting.

In [ ]:
def spec_speedup(gamma, accept_rate, verify_cost=1.15, draft_cost=0.12):
    """
    gamma: draft tokens proposed per round.
    verify_cost: verifier forward pass over gamma+1 positions, relative to one normal step.
    draft_cost: one draft-model step, relative to one verifier step.
    Expected accepted tokens for i.i.d. acceptance = (1 - a^(g+1)) / (1 - a).
    """
    a = accept_rate
    exp_tokens = (1 - a ** (gamma + 1)) / (1 - a) if a < 1 else gamma + 1
    cost = verify_cost + gamma * draft_cost
    return exp_tokens / cost

print(f"{'accept rate':>12}" + "".join(f"{'g=' + str(g):>9}" for g in [2, 4, 6, 8]))
print("-" * 50)
for a in [0.5, 0.65, 0.8, 0.9]:
    row = "".join(f"{spec_speedup(g, a):>9.2f}" for g in [2, 4, 6, 8])
    print(f"{a:>11.0%}" + row)

print()
print("Two things this table shows:")
print("  1. Speedup is dominated by acceptance rate, not by gamma.")
print("  2. Larger gamma HURTS at low acceptance — you pay for drafts you throw away.")
print("     Tune gamma to your measured acceptance rate; do not just raise it.")

## 6. The Serving Stack

You will be asked "what would you actually deploy." As of 2026 three open engines dominate:

| Engine | Distinguishing idea | Strongest fit |
|---|---|---|
| **vLLM** | PagedAttention; widest model coverage; largest community | General-purpose default; high-throughput batch |
| **SGLang** | RadixAttention prefix reuse; structured-generation focus | Agents, shared system prompts, constrained output |
| **TensorRT-LLM** | Ahead-of-time compilation to NVIDIA kernels | NVIDIA-committed teams who can absorb compile time |

All three now do continuous batching, paged KV, tensor parallelism, FP8/INT8/INT4
quantization, streaming, and speculative decoding. **Per-token throughput between them is
generally within 10–20% on the same model and hardware**, and which one leads flips with
workload shape and release cadence. So the honest interview answer is not "engine X is
fastest" — it's a decision framework:

1. **Workload shape.** Heavy shared prefixes (agents, RAG with a fixed corpus prompt) favour
   aggressive prefix reuse. Uniform independent requests favour raw throughput.
2. **Structured output requirements.** If every response must satisfy a schema, prefer an
   engine with first-class constrained decoding.
3. **Hardware commitment.** Compilation-based stacks pay off only if you're staying on one
   vendor.
4. **Operational maturity.** Whatever your team can debug at 3am beats a 15% benchmark win.

Since all of them expose an OpenAI-compatible API, swapping is largely a config change — so
this is a reversible decision, and worth saying so.

**Also worth knowing:** these are fast-moving projects that have shipped critical CVEs,
including remote-code-execution issues in multimodal and disaggregated-serving paths. Pin
versions, track advisories, and don't expose an inference server directly to untrusted input.

### Beyond a single replica
- **Chunked prefill** — split long prefills into pieces so they interleave with in-flight
  decodes instead of stalling them. Protects TPOT when a 100k-token prompt arrives.
- **Disaggregated prefill/decode** — run the two phases on separate hardware pools sized to
  their different bottlenecks. Powerful, and operationally more complex.
- **Router in front** — send easy requests to a small model, hard ones to a large one. Usually
  the largest cost lever available, and it's a *product* decision more than an infra one.

## Common Interview Questions

**Q: Why is decode memory-bandwidth-bound and what follows from that?**
Emitting one token requires reading all model weights and the KV cache from HBM while doing
only ~2×params FLOPs. Accelerators have far more FLOP/s than bytes/s, so the read dominates.
Consequences: batching is the primary throughput lever (weights get read once for the whole
batch), quantization helps roughly in proportion to bytes saved, and single-stream latency is
nearly independent of how much compute you throw at it.

**Q: What does PagedAttention solve?**
Internal fragmentation. Reserving contiguous max-length KV per sequence wastes most of the
cache because real lengths vary. Paging allocates fixed-size blocks on demand with an
indirection table, bounding waste to one partial page per sequence, and it makes prefix
sharing between requests cheap because pages can be referenced by more than one sequence.
The practical result is a large increase in feasible batch size.

**Q: Your agent product has a 4k-token system prompt and p95 TTFT is too high. What do you do?**
Prefix caching first — that prompt is identical on every request, so its KV should be computed
once and reused. Make sure nothing variable (timestamps, shuffled tool ordering, session IDs)
sits before the stable content, since anything that changes near the front invalidates
everything after it. Then chunked prefill so long prompts don't stall decodes, and only then
look at shortening the prompt or switching engines.

**Q: When is speculative decoding a bad idea?**
When the server is already compute-saturated at high batch size — speculation spends extra
FLOPs to buy latency, and if compute is the constraint it lowers aggregate throughput. Also
when the text is high-entropy, since low acceptance means you discard most drafted tokens and
pay for them anyway. It shines on low-batch, latency-sensitive, predictable-output workloads.

**Q: How do you choose between vLLM, SGLang, and TensorRT-LLM?**
Throughput between them is close enough that the benchmark rarely decides it. I'd choose on
workload shape (heavy shared prefixes favour radix-style reuse), structured output needs,
hardware commitment (compiled stacks lock you to a vendor), and what the team can operate.
They all expose OpenAI-compatible APIs, so the decision is reversible — which is worth saying,
because it lowers the stakes and shows you've actually migrated one.

**Q: Where would you cut cost first on an LLM feature?**
Routing, before any infra work. Most traffic is easy; sending all of it to the largest model
is usually the biggest single waste. Then prefix caching, then quantization, then batching
configuration. Distillation of the common path (llm2) often beats all of these but takes
longer to land.

## Key Takeaways
- Prefill is parallel and compute-bound (sets TTFT); decode is sequential and bandwidth-bound (sets TPOT)
- Batching is the dominant throughput lever because it amortizes weight reads across sequences
- Continuous batching has been table stakes since ~2023 — treat it as the baseline, not a win
- PagedAttention removes KV fragmentation; prefix caching removes redundant prefill for shared prompts
- Keep the invariant part of a prompt at the very front so it stays prefix-cacheable
- Weight and KV-cache quantization are independent choices; at long context the cache may dominate
- Speculative decoding is exact, not approximate — but its speedup lives or dies on acceptance rate
- vLLM / SGLang / TensorRT-LLM are within ~10–20% on throughput; choose on workload shape and operability
- Cheapest cost lever is usually routing easy traffic to a smaller model, not tuning the serving stack